# TGF-Net: Temporal Gated Fusion Network
## Advanced Dementia Detection from Speech

### Architecture Overview
| Branch | Input | Process | Output |
|---|---|---|---|
| Audio | `[B, ~3500, 768]` WavLM frames | Conv1d compression → Transformer | `[B, 256]` |
| Text | `[B, C, 768]` RoBERTa chunks | Linear proj → Transformer | `[B, 256]` |
| Fusion | Both `[B, 256]` vectors | Learned gating (soft weights) | `[B, 256]` |
| Head | `[B, 256]` | LayerNorm → Linear → GELU → Linear | `[B, 1]` logit |

### Why this fixes your previous models
- **Baseline**: mean-pooled 3500 frames → single vector → all temporal structure lost
- **Attention model**: 3500 audio frames vs ~8 text chunks = 400× mismatch, attention was overwhelmed
- **TGF-Net**: Conv1d compresses audio to ~109 frames *before* Transformer; gating learns to weight each modality per-sample

---
## 1. Install & Imports

In [1]:
# Uncomment if needed
# !pip install torch scikit-learn numpy

import os
import time
import json
import numpy as np
from pathlib import Path
from typing import Literal

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.3 GB


---
## 2. Configuration
**The cell below auto-detects your project root. Just set `PROJECT_ROOT` to the folder that contains your `data/` directory.**

In [2]:
# ── SET THIS to the folder that contains your data/ directory ─────────────────
# Leave as None to auto-detect (searches current dir and parents for data/splits/)
PROJECT_ROOT = r"D:\Finalyear project 2\fyp-2"
# Example overrides:
# PROJECT_ROOT = r'C:/Users/yourname/dementia_project'
# PROJECT_ROOT = '/home/yourname/dementia_project'

def find_project_root(marker='data/splits/patient_splits.json', start=None):
    """Walk up from start dir until we find the marker file."""
    candidate = Path(start or os.getcwd()).resolve()
    for _ in range(6):  # search up to 6 levels up
        if (candidate / marker).exists():
            return candidate
        candidate = candidate.parent
    return None

if PROJECT_ROOT is None:
    detected = find_project_root()
    if detected:
        PROJECT_ROOT = str(detected)
        print(f'Auto-detected project root: {PROJECT_ROOT}')
    else:
        PROJECT_ROOT = os.getcwd()
        print(f'WARNING: Could not auto-detect project root.')
        print(f'Falling back to current directory: {PROJECT_ROOT}')
        print(f'Set PROJECT_ROOT manually above if paths are wrong.')
else:
    print(f'Using manually set project root: {PROJECT_ROOT}')

# Change working directory so relative imports still work
os.chdir(PROJECT_ROOT)
print(f'Working directory set to: {os.getcwd()}')

# Verify key paths exist
checks = [
    'data/splits/patient_splits.json',
    'data/features/wavlm',
    'data/features/roberta',
]
all_ok = True
for c in checks:
    exists = Path(c).exists()
    status = '✓' if exists else '✗ NOT FOUND'
    print(f'  {status}  {c}')
    if not exists:
        all_ok = False

if not all_ok:
    print('\nFix: Set PROJECT_ROOT above to the folder containing your data/ directory.')

Using manually set project root: D:\Finalyear project 2\fyp-2
Working directory set to: D:\Finalyear project 2\fyp-2
  ✓  data/splits/patient_splits.json
  ✓  data/features/wavlm
  ✓  data/features/roberta


In [21]:
CONFIG = {
    # ── Data paths (relative to PROJECT_ROOT — do not change) ─
    'splits_json':  'data/splits/patient_splits.json',
    'wavlm_dir':    'data/features/wavlm',
    'roberta_dir':  'data/features/roberta',

    # ── Model architecture ────────────────────────────────────
    'hidden_dim':      256,   # internal representation size
    'n_heads':         4,     # transformer attention heads
    'audio_layers':    2,     # transformer layers in audio branch
    'text_layers':     2,     # transformer layers in text branch
    'dropout':         0.3,
    'compress_stride': 8,     # audio temporal downsampling (higher = smaller sequence)

    # ── Training ──────────────────────────────────────────────
    'batch_size':    8,
    'num_workers':   0,
    'epochs':        60,
    'lr':            3e-4,
    'weight_decay':  1e-4,
    'warmup_epochs': 5,

    # ── Focal Loss ────────────────────────────────────────────
    # focal_alpha > 0.5 increases dementia recall
    # raise to 0.7 if dementia recall is still low after training
    'focal_gamma':   2.0,
    'focal_alpha':   0.6,

    # ── Early stopping ────────────────────────────────────────
    'patience': 10,

    # ── Output ────────────────────────────────────────────────
    'checkpoint_dir': 'checkpoints',
    'log_file':       'training_log.json',
}

print('Config loaded.')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

Config loaded.
  splits_json: data/splits/patient_splits.json
  wavlm_dir: data/features/wavlm
  roberta_dir: data/features/roberta
  hidden_dim: 256
  n_heads: 4
  audio_layers: 2
  text_layers: 2
  dropout: 0.3
  compress_stride: 8
  batch_size: 8
  num_workers: 0
  epochs: 60
  lr: 0.0003
  weight_decay: 0.0001
  warmup_epochs: 5
  focal_gamma: 2.0
  focal_alpha: 0.6
  patience: 10
  checkpoint_dir: checkpoints
  log_file: training_log.json


---
## 3. Dataset & DataLoader

In [22]:
class DementiaDataset(Dataset):
    """
    Matches actual patient_splits.json structure:
        splits['dementia']['train']    = ['Richard Gwyn', ...]
        splits['no_dementia']['train'] = ['Angela Lansbury', ...]
    """
    def __init__(
        self,
        split,
        splits_json='data/splits/patient_splits.json',
        wavlm_dir='data/features/wavlm',
        roberta_dir='data/features/roberta',
        max_audio_frames=4000,
        max_text_chunks=32,
    ):
        super().__init__()
        self.wavlm_dir   = Path(wavlm_dir)
        self.roberta_dir = Path(roberta_dir)
        self.max_audio   = max_audio_frames
        self.max_text    = max_text_chunks

        with open(splits_json) as f:
            splits_data = json.load(f)

        label_map = {'dementia': 1, 'no_dementia': 0}
        self.samples = []

        for subfolder, label in label_map.items():
            patients_in_split = splits_data[subfolder].get(split, [])
            for patient in patients_in_split:
                wavlm_dir_p   = self.wavlm_dir   / subfolder / patient
                roberta_dir_p = self.roberta_dir / subfolder / patient
                if not wavlm_dir_p.exists():
                    continue
                for audio_file in sorted(wavlm_dir_p.glob('*.pt')):
                    stem = audio_file.stem
                    roberta_file = roberta_dir_p / f'{stem}.pt'
                    if roberta_file.exists():
                        self.samples.append((audio_file, roberta_file, label))

        dem  = sum(1 for _, _, l in self.samples if l == 1)
        ctrl = len(self.samples) - dem
        print(f'[{split.upper():5s}] {len(self.samples)} samples  |  Dementia: {dem}  Control: {ctrl}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        wavlm_path, roberta_path, label = self.samples[idx]
        audio = torch.load(wavlm_path,   map_location='cpu')
        text  = torch.load(roberta_path, map_location='cpu')
        if audio.dim() == 1: audio = audio.unsqueeze(0)
        if text.dim()  == 1: text  = text.unsqueeze(0)
        audio = audio[: self.max_audio]
        text  = text[: self.max_text]
        return audio, text, torch.tensor(label, dtype=torch.float32)


def collate_fn(batch):
    audios, texts, labels = zip(*batch)
    audio_lengths = torch.tensor([a.shape[0] for a in audios])
    text_lengths  = torch.tensor([t.shape[0] for t in texts])
    audio_padded  = pad_sequence(audios, batch_first=True, padding_value=0.0)
    text_padded   = pad_sequence(texts,  batch_first=True, padding_value=0.0)
    labels        = torch.stack(labels)
    return audio_padded, text_padded, audio_lengths, text_lengths, labels


def build_dataloaders(cfg):
    datasets = {
        split: DementiaDataset(split, cfg['splits_json'], cfg['wavlm_dir'], cfg['roberta_dir'])
        for split in ('train', 'val', 'test')
    }
    loaders = {
        split: DataLoader(
            ds,
            batch_size=cfg['batch_size'],
            shuffle=(split == 'train'),
            collate_fn=collate_fn,
            num_workers=cfg['num_workers'],
            pin_memory=True,
        )
        for split, ds in datasets.items()
    }
    return loaders

---
## 4. Model Architecture

In [23]:
# ── Positional Encoding ───────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-torch.log(torch.tensor(10000.0)) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):  # x: [B, T, D]
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# ── Audio Branch ──────────────────────────────────────────────────────────────
# Input : [B, T, 768]   T ≈ 3000–4000 WavLM frames
# Output: [B, 256]      compact temporal summary
class AudioBranch(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, n_heads=4,
                 n_layers=2, dropout=0.3, compress_stride=8):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)

        # Two strided convolutions: T ≈ 3500 → 437 → 109 frames
        self.compress = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=16,
                      stride=compress_stride, padding=8),
            nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=8, stride=4, padding=4),
            nn.GELU(),
        )
        self.pos_enc = PositionalEncoding(hidden_dim, max_len=256, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=n_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout, batch_first=True,
            norm_first=True,  # pre-norm: more stable on small datasets
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        x = self.input_proj(x)        # [B, T, 256]
        x = x.permute(0, 2, 1)        # [B, 256, T]  → Conv1d expects channels first
        x = self.compress(x)          # [B, 256, ~109]
        x = x.permute(0, 2, 1)        # [B, ~109, 256]
        x = self.pos_enc(x)
        x = self.transformer(x)       # [B, ~109, 256]
        x = self.norm(x)
        return x.mean(dim=1)          # [B, 256]


# ── Text Branch ───────────────────────────────────────────────────────────────
# Input : [B, C, 768]   C = RoBERTa chunks
# Output: [B, 256]
class TextBranch(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, n_heads=4,
                 n_layers=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.pos_enc    = PositionalEncoding(hidden_dim, max_len=64, dropout=dropout)
        encoder_layer   = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=n_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        x = self.input_proj(x)        # [B, C, 256]
        x = self.pos_enc(x)
        x = self.transformer(x)       # [B, C, 256]
        x = self.norm(x)
        return x.mean(dim=1)          # [B, 256]


# ── Gated Fusion ──────────────────────────────────────────────────────────────
# Learns per-sample soft weights: how much to trust audio vs text
class GatedFusion(nn.Module):
    def __init__(self, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
            nn.Softmax(dim=-1),        # weights sum to 1
        )
        self.proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, audio_feat, text_feat):
        combined = torch.cat([audio_feat, text_feat], dim=-1)  # [B, 512]
        weights  = self.gate(combined)                          # [B, 2]
        w_a, w_t = weights[:, 0:1], weights[:, 1:2]
        fused    = w_a * audio_feat + w_t * text_feat
        fused    = self.proj(fused)
        return fused, weights


# ── Full Model: TGF-Net ───────────────────────────────────────────────────────
class TGFNet(nn.Module):
    """
    Temporal Gated Fusion Network for dementia detection.
    forward() returns: (logit [B,1], gate_weights [B,2])
    gate_weights[:, 0] = audio weight, gate_weights[:, 1] = text weight
    """
    def __init__(self, audio_input_dim=768, text_input_dim=768,
                 hidden_dim=256, n_heads=4, audio_layers=2, text_layers=2,
                 dropout=0.3, compress_stride=8):
        super().__init__()
        self.audio_branch = AudioBranch(audio_input_dim, hidden_dim, n_heads,
                                        audio_layers, dropout, compress_stride)
        self.text_branch  = TextBranch(text_input_dim,  hidden_dim, n_heads,
                                       text_layers, dropout)
        self.fusion       = GatedFusion(hidden_dim, dropout)
        self.classifier   = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, audio, text):
        a               = self.audio_branch(audio)
        t               = self.text_branch(text)
        fused, weights  = self.fusion(a, t)
        logit           = self.classifier(fused)
        return logit, weights

In [24]:
# Instantiate and inspect the model
model = TGFNet(
    hidden_dim=CONFIG['hidden_dim'],
    n_heads=CONFIG['n_heads'],
    audio_layers=CONFIG['audio_layers'],
    text_layers=CONFIG['text_layers'],
    dropout=CONFIG['dropout'],
    compress_stride=CONFIG['compress_stride'],
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'TGF-Net trainable parameters: {n_params:,}')
print(model)

TGF-Net trainable parameters: 5,358,339
TGFNet(
  (audio_branch): AudioBranch(
    (input_proj): Linear(in_features=768, out_features=256, bias=True)
    (compress): Sequential(
      (0): Conv1d(256, 256, kernel_size=(16,), stride=(8,), padding=(8,))
      (1): GELU(approximate='none')
      (2): Conv1d(256, 256, kernel_size=(8,), stride=(4,), padding=(4,))
      (3): GELU(approximate='none')
    )
    (pos_enc): PositionalEncoding(
      (dropout): Dropout(p=0.3, inplace=False)
    )
    (transformer): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=1024, bias=True)
          (dropout): Dropout(p=0.3, inplace=False)
          (linear2): Linear(in_features=1024, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps

In [25]:
# Quick sanity check — forward pass with dummy tensors
dummy_audio = torch.randn(2, 3500, 768).to(device)  # batch=2, T=3500, dim=768
dummy_text  = torch.randn(2,    8, 768).to(device)  # batch=2, C=8,    dim=768

logits, weights = model(dummy_audio, dummy_text)
print(f'Output logit shape  : {logits.shape}')    # [2, 1]
print(f'Gate weights shape  : {weights.shape}')   # [2, 2]
print(f'Gate weights (sample 0): audio={weights[0,0].item():.3f}  text={weights[0,1].item():.3f}')
print('Sanity check passed.')

Output logit shape  : torch.Size([2, 1])
Gate weights shape  : torch.Size([2, 2])
Gate weights (sample 0): audio=0.466  text=0.534
Sanity check passed.


---
## 5. Loss, Optimizer & Scheduler

In [26]:
class FocalLoss(nn.Module):
    """
    Binary Focal Loss.
    - gamma: focusing parameter (0 = standard BCE, 2 = standard focal)
    - alpha: weight for positive (dementia) class
              increase to 0.7 if dementia recall is still low
    Reduces gradient from easy correctly-classified examples,
    forcing the model to focus on hard borderline cases.
    """
    def __init__(self, gamma=2.0, alpha=0.6):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        logits  = logits.squeeze(-1)
        probs   = torch.sigmoid(logits)
        bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t     = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss    = alpha_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()


def get_scheduler(optimizer, warmup_epochs, total_epochs):
    """Linear warmup -> cosine decay."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


criterion = FocalLoss(gamma=CONFIG['focal_gamma'], alpha=CONFIG['focal_alpha'])
optimizer = optim.AdamW(model.parameters(),
                        lr=CONFIG['lr'],
                        weight_decay=CONFIG['weight_decay'])
scheduler = get_scheduler(optimizer, CONFIG['warmup_epochs'], CONFIG['epochs'])

print('Loss / optimizer / scheduler ready.')

Loss / optimizer / scheduler ready.


---
## 6. Evaluation Function

In [27]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_probs, all_labels = [], [], []
    gate_log = []

    for audio, text, audio_len, text_len, labels in loader:
        audio  = audio.to(device)
        text   = text.to(device)
        labels = labels.to(device)

        logits, weights = model(audio, text)
        total_loss += criterion(logits, labels).item()

        probs = torch.sigmoid(logits.squeeze(-1))
        preds = (probs > 0.5).long()

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().long().numpy())
        gate_log.append(weights.mean(dim=0).cpu().numpy())

    avg_gate = np.mean(gate_log, axis=0)

    return {
        'loss':       total_loss / len(loader),
        'accuracy':   accuracy_score(all_labels, all_preds),
        'f1':         f1_score(all_labels, all_preds, average='weighted', zero_division=0),
        'f1_dem':     f1_score(all_labels, all_preds, pos_label=1, average='binary', zero_division=0),
        'precision':  precision_score(all_labels, all_preds, average='weighted', zero_division=0),
        'recall':     recall_score(all_labels, all_preds, average='weighted', zero_division=0),
        'recall_dem': recall_score(all_labels, all_preds, pos_label=1, average='binary', zero_division=0),
        'auc':        roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0,
        'gate_audio': float(avg_gate[0]),
        'gate_text':  float(avg_gate[1]),
    }

---
## 7. Training Loop

In [28]:
# ── Load data ─────────────────────────────────────────────────────────────────
loaders = build_dataloaders(CONFIG)

[TRAIN] 249 samples  |  Dementia: 89  Control: 160
[VAL  ] 54 samples  |  Dementia: 23  Control: 31
[TEST ] 53 samples  |  Dementia: 19  Control: 34


In [29]:
Path(CONFIG['checkpoint_dir']).mkdir(parents=True, exist_ok=True)

best_val_f1  = 0.0
patience_ctr = 0
history      = []

print(f"{'Epoch':>6} | {'TrainLoss':>9} | {'ValLoss':>7} | "
      f"{'Acc':>5} | {'F1':>5} | {'DemR':>5} | {'AUC':>5} | "
      f"{'GateA':>6} | {'GateT':>6} | {'LR':>8}")
print('-' * 95)

for epoch in range(1, CONFIG['epochs'] + 1):
    model.train()
    train_loss = 0.0
    t0 = time.time()

    for audio, text, audio_len, text_len, labels in loaders['train']:
        audio  = audio.to(device)
        text   = text.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits, _ = model(audio, text)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    avg_train_loss = train_loss / len(loaders['train'])
    val            = evaluate(model, loaders['val'], criterion, device)
    lr_now         = optimizer.param_groups[0]['lr']

    print(
        f"{epoch:6d} | {avg_train_loss:9.4f} | {val['loss']:7.4f} | "
        f"{val['accuracy']:5.3f} | {val['f1']:5.3f} | {val['recall_dem']:5.3f} | "
        f"{val['auc']:5.3f} | {val['gate_audio']:6.3f} | {val['gate_text']:6.3f} | {lr_now:.2e}"
    )

    history.append({'epoch': epoch, 'train_loss': avg_train_loss,
                    **{f'val_{k}': v for k, v in val.items()}, 'lr': lr_now})

    if val['f1'] > best_val_f1:
        best_val_f1  = val['f1']
        patience_ctr = 0
        torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                    'val_metrics': val},
                   CONFIG['checkpoint_dir'] + '/best_model.pt')
        print(f'  ✓ Best model saved  (Val F1={best_val_f1:.4f})')
    else:
        patience_ctr += 1
        if patience_ctr >= CONFIG['patience']:
            print(f'\nEarly stopping at epoch {epoch}')
            break

print('\nTraining complete.')

 Epoch | TrainLoss | ValLoss |   Acc |    F1 |  DemR |   AUC |  GateA |  GateT |       LR
-----------------------------------------------------------------------------------------------


C:\Users\nlnkr\AppData\Local\Temp\ipykernel_17928\2315513616.py:50: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  audio = torch.load(wavlm_path,   map_location='cpu')
C:\Use

     1 |    0.0893 |  0.0840 | 0.574 | 0.419 | 0.000 | 0.560 |  0.455 |  0.545 | 1.20e-04
  ✓ Best model saved  (Val F1=0.4187)


C:\Users\nlnkr\AppData\Local\Temp\ipykernel_17928\2315513616.py:50: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  audio = torch.load(wavlm_path,   map_location='cpu')
C:\Use

     2 |    0.0852 |  0.0887 | 0.574 | 0.419 | 0.000 | 0.677 |  0.266 |  0.734 | 1.80e-04


...

    21 |    0.0110 |  0.4514 | 0.611 | 0.520 | 0.130 | 0.809 |  0.872 |  0.128 | 2.42e-04


C:\Users\nlnkr\AppData\Local\Temp\ipykernel_17928\2315513616.py:50: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  audio = torch.load(wavlm_path,   map_location='cpu')
C:\Use

    22 |    0.0145 |  0.3116 | 0.722 | 0.692 | 0.391 | 0.686 |  0.917 |  0.083 | 2.35e-04


C:\Users\nlnkr\AppData\Local\Temp\ipykernel_17928\2315513616.py:50: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  audio = torch.load(wavlm_path,   map_location='cpu')
C:\Use

    23 |    0.0088 |  0.4470 | 0.630 | 0.532 | 0.130 | 0.617 |  0.855 |  0.145 | 2.27e-04

Early stopping at epoch 23

Training complete.


---
## 8. Final Test Evaluation

In [30]:
# Load best checkpoint and evaluate on held-out test set
ckpt = torch.load(CONFIG['checkpoint_dir'] + '/best_model.pt', map_location=device)
model.load_state_dict(ckpt['state_dict'])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']}")

test = evaluate(model, loaders['test'], criterion, device)

print('\n' + '='*50)
print('TEST SET RESULTS')
print('='*50)
print(f"  Accuracy         : {test['accuracy']:.4f}")
print(f"  Weighted F1      : {test['f1']:.4f}")
print(f"  Dementia F1      : {test['f1_dem']:.4f}")
print(f"  Dementia Recall  : {test['recall_dem']:.4f}")
print(f"  AUC-ROC          : {test['auc']:.4f}")
print(f"  Gate [Audio={test['gate_audio']:.3f} | Text={test['gate_text']:.3f}]")

C:\Users\nlnkr\AppData\Local\Temp\ipykernel_17928\332285315.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CONFIG['checkpoint_dir'] + '/best_model.pt'

Loaded best checkpoint from epoch 13

TEST SET RESULTS
  Accuracy         : 0.7170
  Weighted F1      : 0.7225
  Dementia F1      : 0.6809
  Dementia Recall  : 0.8421
  AUC-ROC          : 0.7724
  Gate [Audio=0.927 | Text=0.073]


---
## 9. Save Training Log & Plot Curves

In [ ]:
with open(CONFIG['log_file'], 'w') as f:
    json.dump({'config': CONFIG, 'history': history, 'test': test}, f, indent=2)
print(f"Log saved → {CONFIG['log_file']}")

In [ ]:
import matplotlib.pyplot as plt

epochs_ran   = [h['epoch']         for h in history]
train_losses = [h['train_loss']     for h in history]
val_losses   = [h['val_loss']       for h in history]
val_accs     = [h['val_accuracy']   for h in history]
val_f1s      = [h['val_f1']         for h in history]
val_dem_rec  = [h['val_recall_dem'] for h in history]
gate_audios  = [h['val_gate_audio'] for h in history]
gate_texts   = [h['val_gate_text']  for h in history]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('TGF-Net Training Curves', fontsize=14, fontweight='bold')

# Loss
axes[0,0].plot(epochs_ran, train_losses, label='Train Loss')
axes[0,0].plot(epochs_ran, val_losses,   label='Val Loss')
axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].grid(True)

# Accuracy & F1
axes[0,1].plot(epochs_ran, val_accs, label='Val Accuracy')
axes[0,1].plot(epochs_ran, val_f1s,  label='Val Weighted F1')
axes[0,1].set_title('Accuracy & F1'); axes[0,1].legend(); axes[0,1].grid(True)

# Dementia Recall (the key metric)
axes[1,0].plot(epochs_ran, val_dem_rec, color='crimson', label='Dementia Recall')
axes[1,0].axhline(0.39, color='gray', linestyle='--', label='Baseline (0.39)')
axes[1,0].set_title('Dementia Recall'); axes[1,0].legend(); axes[1,0].grid(True)

# Gate weights — shows what the model learned to trust
axes[1,1].plot(epochs_ran, gate_audios, label='Audio gate weight')
axes[1,1].plot(epochs_ran, gate_texts,  label='Text gate weight')
axes[1,1].set_title('Gated Fusion Weights (interpretability)')
axes[1,1].legend(); axes[1,1].grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Plot saved → training_curves.png')